# Linear Regression — Statistical Learning

We estimate a regression line, judge how trustworthy its coefficients are, measure model fit, and extend to multiple predictors, dummies, interactions and shrinkage.

## How this notebook works

<table>
<tr><td style="font-size:1.4em;">🧮</td><td><b style="color:#E37222;">Level 1 · By hand</b></td><td>Do the math yourself on small <b>mock data</b> — plain arithmetic, no libraries.</td></tr>
<tr><td style="font-size:1.4em;">🐍</td><td><b style="color:#0065BD;">Level 2 · With NumPy</b></td><td>Implement the formula with basic <b>NumPy</b> on the real data.</td></tr>
<tr><td style="font-size:1.4em;">📚</td><td><b style="color:#5E6A00;">Level 3 · With libraries</b></td><td>Use <b>pandas / scikit-learn</b> — the practical way.</td></tr>
</table>

Not every topic offers all three levels — the foundational ones do, later applied ones jump straight to the practical level. When stuck, expand a hint: <b>💡 Hint</b> names the functions (with docs); <b>🔑 Stuck?</b> shows them on mock data.

## Setup &mdash; packages and data

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.style.use(os.path.join(os.getcwd(), "..", "..", "configs", "visualisations.mplstyle"))
df = pd.read_csv(os.path.join("..", "..", "data", "1900_2021_DISASTERS.csv"))

df['Total Damages (US$)'] = df["Total Damages ('000 US$)"] * 1000
df['Disaster Type'] = df['Disaster Type'].astype('category')
df.head(5)

,Year,Seq,Glide,Disaster Group,Disaster Subgroup,Disaster Type,Disaster Subtype,Disaster Subsubtype,Event Name,Country,...,No Homeless,Total Affected,Insured Damages ('000 US$),Total Damages ('000 US$),CPI,Adm Level,Admin1 Code,Admin2 Code,Geo Locations,Total Damages (US$)
0,1900,9002,NaN,Natural,Climatological,Drought,Drought,NaN,NaN,Cabo Verde,...,NaN,NaN,NaN,NaN,3.221647,NaN,NaN,NaN,NaN,NaN
1,1900,9001,NaN,Natural,Climatological,Drought,Drought,NaN,NaN,India,...,NaN,NaN,NaN,NaN,3.221647,NaN,NaN,NaN,NaN,NaN
2,1902,12,NaN,Natural,Geophysical,Earthquake,Ground movement,NaN,NaN,Guatemala,...,NaN,NaN,NaN,25000.0,3.350513,NaN,NaN,NaN,NaN,25000000.0
3,1902,3,NaN,Natural,Geophysical,Volcanic activity,Ash fall,NaN,Santa Maria,Guatemala,...,NaN,NaN,NaN,NaN,3.350513,NaN,NaN,NaN,NaN,NaN
4,1902,10,NaN,Natural,Geophysical,Volcanic activity,Ash fall,NaN,Santa Maria,Guatemala,...,NaN,NaN,NaN,NaN,3.350513,NaN,NaN,NaN,NaN,NaN


# Simple linear regression

### Estimating the coefficients
Least squares has a closed form: slope $\hat\beta_1 = \mathrm{Cov}(X,Y)/\mathrm{Var}(X)$, intercept $\hat\beta_0 = \bar y - \hat\beta_1\bar x$.

<div style="background-color:#FCEEE3; border-left:6px solid #E37222; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#E37222;">🧮&nbsp; Level 1 · By hand</b><br><span style="color:#000;">For the mock data <code>x = [1, 2, 3]</code>, <code>y = [2, 4, 5]</code> (so x̄ = 2, ȳ = 11/3), compute the slope and intercept by hand. (Σ(x−x̄)(y−ȳ) = 3, Σ(x−x̄)² = 2.)</span></div>

In [ ]:
xd = [1, 2, 3]
yd = [2, 4, 5]

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'slope b1 = {b1_h:.2f}, intercept b0 = {b0_h:.2f}')

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Implement the closed-form estimates with NumPy on log <code>No Homeless</code> vs log <code>Total Damages</code>, and compare to <code>np.polyfit</code>.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Centre x and y by their means; slope = [`np.sum`](https://numpy.org/doc/stable/reference/generated/numpy.sum.html)`((x-x̄)*(y-ȳ)) / np.sum((x-x̄)**2)`; check against [`np.polyfit`](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html)`(x, y, 1)`.

</details>

In [ ]:
x_axis = 'No Homeless'
y_axis = 'Total Damages (US$)'
mask = (df[x_axis] > 0) & (df[y_axis] > 0)
x = np.log10(df.loc[mask, x_axis])
y = np.log10(df.loc[mask, y_axis])

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f"your fit:    b0 = {b0:.4f},  b1 = {b1:.4f}")
print(f"np.polyfit:  b0 = {np.polyfit(x, y, 1)[1]:.4f},  b1 = {np.polyfit(x, y, 1)[0]:.4f}")
plt.scatter(x, y, s=20, marker='o', alpha=1, edgecolors='w', linewidth=0.5)
x_fit = np.linspace(x.min(), x.max(), 100)
plt.plot(x_fit, b0 + b1 * x_fit, color='red', linewidth=2, label='OLS fit')
plt.xticks(ticks=np.log10([1e0, 1e1, 1e2, 1e3, 1e4, 1e5, 1e6]), labels=['$10^0$', '$10^1$', '$10^2$', '$10^3$', '$10^4$', '$10^5$', '$10^6$'])
plt.yticks(ticks=np.log10([1e4, 1e6, 1e8, 1e10, 1e12]), labels=['$10^4$', '$10^6$', '$10^8$', '$10^{10}$', '$10^{12}$'])
plt.legend(); plt.xlabel(x_axis); plt.ylabel(y_axis); plt.title('OLS fit'); plt.show()

### Accuracy of the coefficients — standard error by bootstrap

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Estimate the standard error of the slope by <b>bootstrapping</b>: resample with replacement 1000 times, refit, take the std of the slopes. Compare to statsmodels' analytic SE.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Draw bootstrap indices with [`np.random.default_rng`](https://numpy.org/doc/stable/reference/generated/numpy.random.default_rng.html)`().integers(0, n, n)`; refit with [`np.polyfit`](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html); the analytic SE is `ref.bse.iloc[1]` from [`sm.OLS`](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html).

</details>

In [ ]:
import statsmodels.api as sm
Xc = sm.add_constant(x)
ref = sm.OLS(y, Xc).fit()
se_analytic = ref.bse.iloc[1]
B = 1000
rng = np.random.default_rng(0)
n = len(x)
xv, yv = x.values, y.values
slopes = np.empty(B)

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'bootstrap SE(b1) = {slopes.std(ddof=1):.4f}')
print(f'analytic  SE(b1) = {se_analytic:.4f}')

### Hypothesis test on the slope

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Test H₀: β₁ = 0 by computing the t-statistic t = β̂₁ / SE(β̂₁) and its two-sided p-value, then compare to statsmodels.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

`ref.params.iloc[1]`, `ref.bse.iloc[1]`, `ref.df_resid` give the pieces; two-sided p-value = `2 * scipy.stats.t.sf(|t|, dof)` ([scipy.stats.t](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html)).

</details>

In [ ]:
from scipy import stats
b1_hat = ref.params.iloc[1]
se_b1 = ref.bse.iloc[1]
dof = ref.df_resid

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'your t = {t_stat:.3f}, p = {p_value:.3e}')
print(f'sm   t = {ref.tvalues.iloc[1]:.3f}, p = {ref.pvalues.iloc[1]:.3e}')

### Accuracy of the model — R²
Total variation splits as TSS = RSS + ESS, and $R^2 = 1 - \mathrm{RSS}/\mathrm{TSS}$.

<div style="background-color:#FCEEE3; border-left:6px solid #E37222; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#E37222;">🧮&nbsp; Level 1 · By hand</b><br><span style="color:#000;">For <code>y = [2, 4, 6]</code> with fitted <code>ŷ = [2, 4, 5]</code> (so ȳ = 4), compute TSS = Σ(y−ȳ)², RSS = Σ(y−ŷ)² and R² = 1 − RSS/TSS by hand.</span></div>

In [ ]:
yt = [2.0, 4.0, 6.0]
yh = [2.0, 4.0, 5.0]   # ȳ = 4

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'TSS = {TSS_h}, RSS = {RSS_h}, R² = {R2_h}')

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">On the real fit, compute TSS, RSS, ESS with NumPy, verify TSS = RSS + ESS, and compute R². Compare to statsmodels' <code>rsquared</code>.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

`y_hat = ref.fittedvalues`; use [`np.sum`](https://numpy.org/doc/stable/reference/generated/numpy.sum.html) for each sum of squares; `R2 = 1 - RSS/TSS`.

</details>

In [ ]:
y_hat = ref.fittedvalues

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'TSS = {TSS:.3f}')
print(f'RSS + ESS = {RSS + ESS:.3f}  (should equal TSS)')
print(f'your R2 = {R2:.4f},  statsmodels R2 = {ref.rsquared:.4f}')

# Multiple linear regression

### The closed-form matrix solution

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Implement the normal-equation solution β̂ = (XᵀX)⁻¹Xᵀy with NumPy (prepend a column of ones), and compare to statsmodels.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Stack an intercept with [`np.column_stack`](https://numpy.org/doc/stable/reference/generated/numpy.column_stack.html)`([np.ones(n), X])`; solve with [`np.linalg.inv`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.inv.html)`(Xmat.T @ Xmat) @ Xmat.T @ y`.

</details>

In [ ]:
features = ["Total Deaths", "No Homeless", "No Injured"]
target = "Total Damages (US$)"
Xm = np.log10(df[features])
ym = np.log10(df[target])
valid = ~(Xm.isnull().any(axis=1) | Xm.isin([np.inf, -np.inf]).any(axis=1) | ym.isnull() | ym.isin([np.inf, -np.inf]))
Xm, ym = Xm[valid].astype(float), ym[valid].astype(float)
Xmat = np.column_stack([np.ones(len(Xm)), Xm.values])
yvec = ym.values

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

sm_params = sm.OLS(yvec, Xmat).fit().params
print('your beta_hat:', np.round(beta_hat, 4))
print('statsmodels: ', np.round(sm_params, 4))

### Joint significance — the F-test

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Compute the overall F-statistic F = ((TSS−RSS)/p) / (RSS/(n−p−1)) for the multiple model, and compare to statsmodels' <code>fvalue</code>.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

`full = sm.OLS(yvec, Xmat).fit()`; `y_hat_m = full.fittedvalues`; `p = len(features)`; build TSS/RSS with [`np.sum`](https://numpy.org/doc/stable/reference/generated/numpy.sum.html).

</details>

In [ ]:
full = sm.OLS(yvec, Xmat).fit()
y_hat_m = full.fittedvalues
n = len(yvec)
p = len(features)

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'your F = {F:.2f}')
print(f'sm   F = {full.fvalue:.2f}')

# Extensions

### Qualitative predictors — dummy variables

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Add a 0/1 <code>is_wildfire</code> dummy to a regression of log-damage on log-homeless. Fit it, read the intercept shift β₂, and plot the two parallel lines.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Build the design with [`sm.add_constant`](https://www.statsmodels.org/stable/generated/statsmodels.tools.tools.add_constant.html) over a DataFrame of `logHomeless` and `is_wildfire`; fit with [`sm.OLS`](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html).

</details>

In [ ]:
x_axis = 'No Homeless'
y_axis = 'Total Damages (US$)'
mask = (df[x_axis] > 0) & (df[y_axis] > 0)
sub = df.loc[mask].copy()
xx = np.log10(sub[x_axis]); yy = np.log10(sub[y_axis])
is_wildfire = (sub['Disaster Type'] == 'Wildfire').astype(float)
Xd = sm.add_constant(pd.DataFrame({'logHomeless': xx.values, 'is_wildfire': is_wildfire.values}))

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'intercept shift for wildfires:  b2 = {b2:.3f}')
xf = np.linspace(xx.min(), xx.max(), 100)
plt.scatter(xx, yy, s=15, alpha=0.4, color='#2C6FA6', edgecolors='w', linewidth=0.3)
plt.plot(xf, b0 + b1 * xf, color='#2C6FA6', lw=2, label='other')
plt.plot(xf, (b0 + b2) + b1 * xf, color='#C0392B', lw=2, label='wildfire')
plt.xticks(ticks=np.log10([1e0, 1e1, 1e2, 1e3, 1e4, 1e5, 1e6]), labels=['$10^0$', '$10^1$', '$10^2$', '$10^3$', '$10^4$', '$10^5$', '$10^6$'])
plt.yticks(ticks=np.log10([1e4, 1e6, 1e8, 1e10, 1e12]), labels=['$10^4$', '$10^6$', '$10^8$', '$10^{10}$', '$10^{12}$'])
plt.legend(); plt.xlabel(x_axis); plt.ylabel(y_axis); plt.title('Additive wildfire dummy'); plt.show()

### Modeling feature interaction

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Add a product term <code>is_wildfire × logHomeless</code> so the slope can differ. Fit it and report the slope for wildfires (β₁+β₃) vs. all others (β₁).</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Add an `interaction` column = `is_wildfire * logHomeless` to the design matrix; the wildfire slope is `b1 + b3`.

</details>

In [ ]:
Xi = sm.add_constant(pd.DataFrame({
    'logHomeless': xx.values,
    'is_wildfire': is_wildfire.values,
    'interaction': is_wildfire.values * xx.values,
}))

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'slope (other)    = {b1:.3f}')
print(f'slope (wildfire) = {b1 + b3:.3f}   (= b1 + b3, with b3 = {b3:.3f})')

### Coefficient shrinkage — Ridge and Lasso

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Standardize five predictors, then fit Ridge and Lasso across a grid of λ and plot the coefficient paths. Which predictors does Lasso drive to exactly zero?</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Scale with [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html); fit [`Ridge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) and [`Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) for each alpha in [`np.logspace`](https://numpy.org/doc/stable/reference/generated/numpy.logspace.html)`(-3, 4, 60)`.

</details>

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
features = ["Total Deaths", "No Injured", "No Affected", "No Homeless", "Total Affected"]
target = "Total Damages (US$)"
d = df.dropna(subset=[target]).copy()
d = d[d[target] > 0]
Xs = StandardScaler().fit_transform(np.log1p(d[features].apply(pd.to_numeric, errors='coerce').fillna(0.0).astype(float)))
ys = np.log10(d[target].astype(float))
ys = ys - ys.mean()
lambdas = np.logspace(-3, 4, 60)

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, paths, t in [(axes[0], ridge_paths, 'Ridge (L2)'), (axes[1], lasso_paths, 'Lasso (L1)')]:
    for j, f in enumerate(features):
        ax.plot(lambdas, paths[:, j], lw=2, label=f)
    ax.set_xscale('log'); ax.axhline(0, color='grey', ls='--', lw=0.8)
    ax.set_xlabel(r'$\lambda$ (penalty strength)'); ax.set_title(t)
axes[0].set_ylabel('standardized coefficient'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()